## FPGA Block Matrix Multiplication Wrapper Function

In [1]:
from pynq import Overlay
from pynq import allocate
import numpy as np

In [2]:
def init_fpga(bitstream_path='/home/ubuntu/workspace/systolic_mmult.bit'):
    global overlay, dma, dma_send, dma_recv
    overlay = Overlay(bitstream_path)
    dma = overlay.axi_dma_0
    dma_send = dma.sendchannel
    dma_recv = dma.recvchannel

def pack_matrix(mat):
    """
    Packs a 4x4 matrix of INT8 values into a 1D array of 4 uint32 words.
    Each row (4 x 8-bit elements) is packed as: [elem3 << 24 | elem2 << 16 | elem1 << 8 | elem0]
    """
    # Cast to uint32 to ensure bitwise operations don't overflow signed representation
    mat_u32 = np.asarray(mat, dtype=np.uint8).astype(np.uint32)
    
    shifts = np.array([0, 8, 16, 24], dtype=np.uint32)
    # Shift columns and bitwise-OR along axis 1 (row-wise)
    return np.bitwise_or.reduce(mat_u32 << shifts, axis=1)

def unpack_matrix(buf):
    """
    Unpacks a 1D buffer of 8 uint64 words into a 4x4 matrix of INT32 accumulators.
    Each 64-bit word contains two 32-bit signed values:
      - Low 32 bits (bits 0..31)
      - High 32 bits (bits 32..63)
    8 words x 2 values = 16 elements (4x4 matrix).
    """
    buf64 = np.asarray(buf, dtype=np.uint64)
    
    # Extract lower 32 bits and upper 32 bits from each 64-bit word
    low_32  = (buf64 & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    high_32 = ((buf64 >> 32) & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    
    # Interleave low and high 32-bit words to reconstruct the 16 elements
    c_flat = np.empty(16, dtype=np.int32)
    c_flat[0::2] = low_32
    c_flat[1::2] = high_32
    
    return c_flat.reshape(4, 4)

def systolic_mmult(a, b):
    """
    Executes 4x4 matrix multiplication A @ B on hardware.
    - Inputs: a [4x4], b [4x4] (INT8)
    - Output: c [4x4] (INT32)
    """
    # 1. Allocate contiguous memory for DMA
    inp_buf = allocate(shape=(8,), dtype=np.uint32)  # 4 words B + 4 words A
    out_buf = allocate(shape=(8,), dtype=np.uint64)  # 8 words (16 x 32-bit outputs)
    
    # 2. Pack matrices: B first, then A (matches your working input_buffer)
    packed_b = pack_matrix(b)
    packed_a = pack_matrix(a)
    inp_buf[:4] = packed_b
    inp_buf[4:] = packed_a
    
    # 3. DMA Transfer
    dma_send.transfer(inp_buf)
    dma_recv.transfer(out_buf)
    dma_send.wait()
    dma_recv.wait()
    
    # 4. Unpack 64-bit output buffer into 4x4 INT32 matrix
    c = unpack_matrix(out_buf)
    
    # 5. Free DMA memory buffers
    del inp_buf, out_buf
    
    return c

In [3]:
def block_matmul(a, b, block_size=4):
    # Get the dimensions of the input matrices
    m, n_a = a.shape
    n_b, p = b.shape
    if n_a != n_b:
        raise ValueError("Matrix dimensions must be compatible for multiplication")

    # Padding matrix a
    num_row_pad_a = int(np.ceil(m / block_size) * block_size - m)
    num_col_pad_a = int(np.ceil(n_a / block_size) * block_size - n_a)
    padded_row_a = np.vstack([a, np.zeros((num_row_pad_a, a.shape[1]))])
    padded_a = np.hstack([padded_row_a, np.zeros((padded_row_a.shape[0], num_col_pad_a))])

    # Padding matrix b
    num_row_pad_b = int(np.ceil(n_b / block_size) * block_size - n_b)
    num_col_pad_b = int(np.ceil(p / block_size) * block_size - p)
    padded_row_b = np.vstack([b, np.zeros((num_row_pad_b, b.shape[1]))])
    padded_b = np.hstack([padded_row_b, np.zeros((padded_row_b.shape[0], num_col_pad_b))])
    
    # Result matrix c with padding
    c = np.zeros((padded_a.shape[0], padded_b.shape[1]))

    # Iterate over blocks (using padded matrices!)
    for i in range(0, padded_a.shape[0], block_size):
        for j in range(0, padded_b.shape[1], block_size):
            for k in range(0, padded_a.shape[1], block_size):
                A_block = padded_a[i:i+block_size, k:k+block_size]
                B_block = padded_b[k:k+block_size, j:j+block_size]

                # Multiply the blocks
                C_block = systolic_mmult(A_block.astype(np.int8), B_block.astype(np.int8))

                # Accumulate the result
                c[i:i+block_size, j:j+block_size] += C_block

    # Remove padding
    return c[:m, :p]

In [4]:
init_fpga()

In [5]:
A = np.random.randint(-128, 127, size=(14, 10))
B = np.random.randint(-128, 127, size=(10, 7))

In [6]:
A

array([[  81, -113,  -62,   54,   -2,  116,   22,  -81,  -34, -127],
       [ -11,  -66,   -7,  -88,   84,   23,  -60,  -68, -120,  118],
       [ -34,   25,  123,  -57,   73,  -86,  -32, -106,  107,   14],
       [ -75,   23,    6, -128,  -48,   11,  -77,   76,  120,   31],
       [  57,   28,  -77,  101,   34,   68,  -94,   47, -111,   54],
       [  82,  108,   41,  -89,  -20,   76,   41,    0,  -67,  -20],
       [  88,   -6,    1,  -85,  103,  -51,  -80,  -61,  -78,   41],
       [ 124,    1, -128,  -35,  -92,    3,   44,   31,   11,  -61],
       [ -35,   -6,   96,   71,    4,  -38,   53, -108,   -1,  -70],
       [  37,  106,  108,  126,   41,  119,  -93,   -6,   68,   22],
       [-124, -116,  -96,  -82,  -83,  -73,  -47,   48,   99,   -5],
       [   6,   75,   65, -111,   49,  114,  106,  122,   41,  -76],
       [ 122,  -42,   -8,  118,   -2, -108,   34, -116,   24,  -53],
       [  37,   79,   65,  109,   67, -109,   36,   99,   54,  -73]])

In [7]:
B

array([[ -11,   96,   12,  -29,   51,  -35, -128],
       [ 105,  -11,   36,   77,   19,  -89,  -84],
       [ 118,  -22, -123,  -80,  117,   38,   43],
       [  22,  -83,   65,    9,   58,  -37,  108],
       [   6,  111,   52,  -39,  -65, -105,   -7],
       [ -19,   68, -119,   45,   84, -117,   99],
       [ 120,   46,  -74,  -66,   72,  -49,   81],
       [ -22,  -75,   46,  111,  104,  -72,  -70],
       [  75,   -9,  119,  122,   37,  -23,  -64],
       [-127,  104,   38,  -83,  -26,   35,   68]])

In [8]:
C = block_matmul(A, B)
C

array([[ -3099.,   7752., -20094.,  -4356.,   2940.,  -9403.,  14780.],
       [-39194.,  33708., -14220., -35258., -30166.,  12464.,  14440.],
       [ 23070.,   7712.,   6445., -11921., -11876.,  15374.,  -8708.],
       [ -5214.,  -8639.,  11717.,  30266.,    -72.,   5921., -21602.],
       [-33136.,   5639.,   9365.,   2703.,  -3601., -17011.,   4315.],
       [ 14189.,  16526., -27807.,  -3163.,  14546., -15591., -13504.],
       [-21078.,  29373.,   2007., -24581., -27474.,   7330., -20097.],
       [ -4572.,    862.,   7013.,  17071.,   3954.,  -5465., -27757.],
       [ 30942., -10172., -14768., -18152.,   4366.,   9558.,  20147.],
       [ 15502.,     43.,   2671.,  15281.,  25800., -25320.,   6137.],
       [-21695., -23060.,  22462.,  22481., -19852.,  27701.,  -7857.],
       [ 33928.,   8178., -23697.,  20477.,  33102., -36372., -13064.],
       [ 13103.,   -474.,  14344., -17645.,  -4960.,  11923.,  -4632.],
       [ 35892., -21616.,  24375.,  14533.,  20588., -16860., -2

In [9]:
D = np.matmul(A, B)
D

array([[ -3099,   7752, -20094,  -4356,   2940,  -9403,  14780],
       [-39194,  33708, -14220, -35258, -30166,  12464,  14440],
       [ 23070,   7712,   6445, -11921, -11876,  15374,  -8708],
       [ -5214,  -8639,  11717,  30266,    -72,   5921, -21602],
       [-33136,   5639,   9365,   2703,  -3601, -17011,   4315],
       [ 14189,  16526, -27807,  -3163,  14546, -15591, -13504],
       [-21078,  29373,   2007, -24581, -27474,   7330, -20097],
       [ -4572,    862,   7013,  17071,   3954,  -5465, -27757],
       [ 30942, -10172, -14768, -18152,   4366,   9558,  20147],
       [ 15502,     43,   2671,  15281,  25800, -25320,   6137],
       [-21695, -23060,  22462,  22481, -19852,  27701,  -7857],
       [ 33928,   8178, -23697,  20477,  33102, -36372, -13064],
       [ 13103,   -474,  14344, -17645,  -4960,  11923,  -4632],
       [ 35892, -21616,  24375,  14533,  20588, -16860, -20499]])

In [10]:
np.array_equal(C, D)

True